# Task 1 - LoRA fine-tuning: Banking77 intent -> strict JSON
Model: Qwen2.5-0.5B-Instruct | GPU: free Colab/Kaggle T4
Task: given a customer banking message, output ONLY {"intent": "<one of 77 labels>"}
Metrics: strict accuracy (valid JSON AND correct label), JSON-valid rate, in-label-set rate, macro-F1

In [ ]:
!pip -q install peft datasets
# If Colab shows a "Restart session" prompt, click it, then continue from the NEXT cell (don't re-run this one).

In [ ]:
import json, random, re, time
from collections import Counter
import numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForCausalLM, TrainingArguments,
                          Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TRAIN_PER_CLASS = 20   # 77 * 20 = 1540 train examples (fast run)
VAL_PER_CLASS = 1      # 77 val examples
TEST_PER_CLASS = 2     # 154 held-out test examples, from the OFFICIAL test split
MAX_LEN = 768
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()   # T4 -> False -> fp16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16)

In [ ]:
# Data: load the original CSVs straight from PolyAI's GitHub (avoids HF loader issues)
BASE = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
train_full = pd.read_csv(BASE + "train.csv")
test_full = pd.read_csv(BASE + "test.csv")
print(train_full.columns.tolist(), len(train_full), len(test_full))   # expect ['text', 'category']

LABELS = sorted(train_full["category"].unique())
assert len(LABELS) == 77

def stratified(df, n, seed):
    return df.groupby("category", group_keys=False).apply(lambda g: g.sample(n=min(n, len(g)), random_state=seed))

train_df = stratified(train_full, TRAIN_PER_CLASS, SEED)
rest = train_full.drop(train_df.index)
val_df = stratified(rest, VAL_PER_CLASS, SEED)
test_df = stratified(test_full, TEST_PER_CLASS, SEED).reset_index(drop=True)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
# Prompt format (IDENTICAL for base and fine-tuned model, so the comparison is fair)
SYSTEM = ("You are a banking support intent classifier. Reply with ONLY a JSON object of the form "
          '{"intent": "<label>"} where <label> is exactly one of the allowed labels.\n'
          "Allowed labels: " + ", ".join(LABELS))

def build_messages(text):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]

def target_str(label):
    return json.dumps({"intent": label})

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def prompt_text(text):
    return tok.apply_chat_template(build_messages(text), add_generation_prompt=True, tokenize=False)

print(prompt_text("I still haven't received my card"))
print("prompt tokens:", len(tok(prompt_text("I still haven't received my card"), add_special_tokens=False)["input_ids"]))

In [ ]:
# Tokenise with loss masking: loss is computed ONLY on the answer tokens, not the long prompt
def encode(row):
    p_ids = tok(prompt_text(row["text"]), add_special_tokens=False)["input_ids"]
    a_ids = tok(target_str(row["category"]) + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_ds = Dataset.from_pandas(train_df[["text", "category"]].reset_index(drop=True)).map(encode, remove_columns=["text", "category"])
val_ds = Dataset.from_pandas(val_df[["text", "category"]].reset_index(drop=True)).map(encode, remove_columns=["text", "category"])
print("max seq len:", max(len(x) for x in train_ds["input_ids"]))

In [ ]:
# Inference + metrics
@torch.no_grad()
def predict(model, texts, bs=16, max_new_tokens=32):
    tok.padding_side = "left"          # left-pad for batched generation
    model.eval()
    outs = []
    for i in range(0, len(texts), bs):
        enc = tok([prompt_text(t) for t in texts[i:i + bs]], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return outs

def parse(raw):
    """returns (intent_strict, intent_lenient). strict = whole output must be valid JSON."""
    strict = None
    try:
        obj = json.loads(raw.strip())
        if isinstance(obj, dict):
            strict = obj.get("intent")
    except Exception:
        pass
    lenient = strict
    if lenient is None:                 # lenient: pull the first {...} out of chatty / ```json output
        m = re.search(r"\{.*?\}", raw, re.S)
        if m:
            try:
                lenient = json.loads(m.group(0)).get("intent")
            except Exception:
                pass
    return strict, lenient

def evaluate(raw_outputs, gold):
    strict, lenient = zip(*[parse(r) for r in raw_outputs])
    n = len(gold)
    res = {
        "n": n,
        "json_valid_rate": sum(s is not None for s in strict) / n,
        "in_label_set_rate": sum(l in LABELS for l in lenient) / n,
        "strict_accuracy": sum(s == g for s, g in zip(strict, gold)) / n,
        "lenient_accuracy": sum(l == g for l, g in zip(lenient, gold)) / n,
        "macro_f1_lenient": f1_score(gold, [l if l in LABELS else "INVALID" for l in lenient],
                                     labels=LABELS, average="macro", zero_division=0),
    }
    return res, list(lenient)

test_texts, test_gold = test_df["text"].tolist(), test_df["category"].tolist()

In [ ]:
# 1) BASE model on held-out test set
dtype = torch.bfloat16 if BF16 else torch.float16
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="cuda")
t0 = time.time()
base_raw = predict(base, test_texts)
base_res, base_pred = evaluate(base_raw, test_gold)
base_res["eval_seconds"] = round(time.time() - t0, 1)
print("BASE:", json.dumps(base_res, indent=2))
for r in base_raw[:5]:
    print(repr(r))
del base; torch.cuda.empty_cache()

In [ ]:
# 2) LoRA setup
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="cuda")
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
for n, p in model.named_parameters():   # keep trainable LoRA weights in fp32 (needed for fp16 AMP on T4)
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

In [ ]:
# 3) Train
tok.padding_side = "right"
args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch 16
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.0,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=not BF16,
    bf16=BF16,
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
train_minutes = round((time.time() - t0) / 60, 1)
peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2)
print("train minutes:", train_minutes, "| peak GPU mem (GB):", peak_gb)
log = pd.DataFrame(trainer.state.log_history)
print(log[[c for c in ["step", "loss", "eval_loss", "epoch"] if c in log]].to_string())

In [ ]:
# 4) FINE-TUNED model on the same held-out test set
ft_raw = predict(model, test_texts)
ft_res, ft_pred = evaluate(ft_raw, test_gold)
print("FINE-TUNED:", json.dumps(ft_res, indent=2))

summary = pd.DataFrame([base_res, ft_res], index=["base", "lora"]).drop(columns=["eval_seconds"], errors="ignore")
print(summary.round(3).to_string())

In [ ]:
# 5) Failure analysis: where does the fine-tuned model still go wrong?
fails = pd.DataFrame({"text": test_texts, "gold": test_gold, "pred": ft_pred, "raw": ft_raw})
fails = fails[fails.gold != fails.pred]
print(len(fails), "failures")
print(Counter(zip(fails.gold, fails.pred)).most_common(10))   # most-confused label pairs
pd.set_option("display.max_colwidth", 120)
print(fails[["text", "gold", "pred"]].to_string())

In [ ]:
# 6) Save everything you need for the report
import os
os.makedirs("results", exist_ok=True)
model.save_pretrained("results/lora_adapter")
json.dump({"base": base_res, "lora": ft_res, "train_minutes": train_minutes, "peak_gpu_gb": peak_gb,
           "lora_config": {"r": lora_cfg.r, "alpha": lora_cfg.lora_alpha, "dropout": lora_cfg.lora_dropout,
                           "target_modules": sorted(lora_cfg.target_modules)},
           "lr": args.learning_rate, "epochs": args.num_train_epochs,
           "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_df)},
          open("results/metrics.json", "w"), indent=2)
pd.DataFrame({"text": test_texts, "gold": test_gold, "base_raw": base_raw, "lora_raw": ft_raw}).to_csv(
    "results/test_predictions.csv", index=False)
log.to_csv("results/train_log.csv", index=False)
print(open("results/metrics.json").read())

In [ ]:
# 7) Download results (Colab). On Kaggle: grab results.zip from the Output panel instead.
!zip -qr results.zip results
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Not on Colab - results.zip is in the working directory")